In [1]:
# Install all necessary libraries, including the new LAMOST/Gaia map
!pip install dustmaps astropy pandas numpy dustmaps3d -q
print("Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.8/754.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.2/193.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 25.5 MB/s eta 0:00:00
Libraries installed successfully


In [ ]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
import dustmaps.bayestar
from dustmaps.bayestar import BayestarQuery
from dustmaps3d import dustmaps3d # The exact import requested by the README
from google.colab import files

def generate_dust(pulsar_name, ra, dec, max_dist_pc=5000, step_pc=10):
    print(f"\nStarting three-dimensional extraction for {pulsar_name}...")

    # Distance vector in parsecs
    distances = np.arange(10, max_dist_pc + step_pc, step_pc)

    # The SkyCoord object understands where the pulsar is
    coords = SkyCoord(ra*u.deg, dec*u.deg, distance=distances*u.pc, frame='icrs')

    # PATH A: NORTHERN HEMISPHERE (Declination > -30°) -> BAYESTAR
    if dec > -30.0:
        print("➡ Northern Hemisphere detected. Using Bayestar19...")
        dustmaps.bayestar.fetch() # Downloads only the first time
        bayestar = BayestarQuery(max_samples=1)
        reddening = bayestar(coords, mode='median')

        # Conversion to Av from Green et al. 2019
        av_values = 2.742 * reddening

    # PATH B: SOUTHERN HEMISPHERE (Declination <= -30°) -> LAMOST/GAIA
    else:
        print("➡ Southern Hemisphere detected. Using dustmaps3d (Wang et al. 2025)...")
        # Convert RA/DEC to Galactic (l, b) as required by the README
        l_arr = np.full(len(distances), coords.galactic.l.degree)
        b_arr = np.full(len(distances), coords.galactic.b.degree)

        # The map requires distances in kpc, not pc
        d_kpc = distances / 1000.0

        # Query the three-dimensional map
        ebv, dust, sigma, max_d = dustmaps3d(l_arr, b_arr, d_kpc)

        # Convert E(B-V) to Av (assuming Rv = 3.1)
        av_values = 3.1 * ebv

    av_values = np.nan_to_num(av_values, nan=0.0)
    # Save it in kpc as your original OOP code expects
    df_av = pd.DataFrame({'Distance(kpc)': distances / 1000.0, 'Extinction(Av)': av_values})

    filename = f"Av_profile_{pulsar_name}.txt"
    df_av.to_csv(filename, sep=' ', index=False, header=False)

    print(f"Success! Downloading '{filename}' to your PC...")
    files.download(filename)

generate_dust("J1124-5916", ra=171.180763, dec=-59.265272, max_dist_pc=5000)

/usr/local/lib/python3.12/dist-packages/dustmaps/config.py:74: ConfigWarning: Configuration file not found:

    /root/.dustmapsrc

To create a new configuration file in the default location, run the following python code:

    from dustmaps.config import config
    config.reset()

Note that this will delete your configuration! For example, if you have specified a data directory, then dustmaps will forget about its location.
  warn(('Configuration file not found:\n\n'



Starting three-dimensional extraction for J1124-5916...
➡ Southern Hemisphere detected. Using dustmaps3d (Wang et al. 2025)...
[dustmaps3d] Downloading data_v3.fits.gz (~400MB)...
[dustmaps3d] Trying primary source (GitHub)...
[dustmaps3d] Starting download from: https://github.com/Grapeknight/dustmaps3d/releases/download/v3/data_v3.fits.gz


data_v3.fits.gz: 100%|██████████| 401M/401M [00:03<00:00, 134MB/s]


[dustmaps3d] ✅ Data has been saved to: /root/.local/share/dustmaps3d/data_v3.fits.gz
[dustmaps3d] Extracting /root/.local/share/dustmaps3d/data_v3.fits.gz ...
[dustmaps3d] ✅ Uncompressed to: /root/.local/share/dustmaps3d/data_v3.fits
[dustmaps3d] ✅ Data ready at: /root/.local/share/dustmaps3d/data_v3.fits
Success! Downloading 'Av_profile_J1124-5916.txt' to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>